# **How Transformer LLMs Work**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [Introduction](#Introduction)
- [Understanding Language Models: Language as a Bag-of-Words](#Understanding-Language-Models:-Language-as-a-Bag-of-Words)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Introduction

Welcome to **How Transformer LLMs Work**. In this course, you learn about the main components of the LLM transformer architecture that has transformed the field of language processing. As you leave this course, you will be able to read through papers describing models and understand the details that are used to describe these architectures. And these intuitions will help you use LLMs better too.

Let me introduce the main topic of this course the transformer. The transformer architecture was first introduced in the 2017 paper, **Attention is All You Need** by Ashish Vaswani and others for machine translation tasks. The idea was to input an English sentence and have the network output a German sentence. The same architecture tend to be great at inputting a prompt and outputting a response to that prompt like a question and the answer to that question. And so this helped herald the early rise of large language models.

The original transformer architecture consisted of two main parts: an **Encoder** and **Decoder**. Consider translating English into German, the encoder preprocess the entire input English text to extract the context needed to perform the translation. Then the decoder uses the encoder context to generate the German. The encoder and the decoder form the basis for the models used in many language models today.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-01.png" width="800">

The encoder model provides rich, context-sensitive representations of the input text, and is the basis for the BERT model and most of the embedding models used in RAG applications. The decoder model performs text generation tasks such as summarizing text, writing code, answering questions, and is the basis for most popular LLMs, such as those from OpenAI, Anthropic, Cohere, and Meta.

**Course Outline**

* You first delve into recent developments in LLMs to see how a sequence of increasingly sophisticated building blocks led to the modern transformer.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-02.png" width="800">

* You then learn about tokenization, which consists of taking text and breaking it down into tokens that comprise words or word fragments. They can then be fed into the LLM.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-03.png" width="800">

* After that, you gain intuition about how the transformer network works, focusing on the decoder-only models. A generative model takes in a text prompt, and it generates a text in response by generating one token at a time.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-04.png" width="800">

    Here's how the generation process works. The model starts by mapping each input token into an embedding vector that captures the meaning of that token. After that, the model parses these token embeddings through a stack of transformer blocks, where each block is a specific neural network architecture that is designed to learn flexibly from data and also scale well on GPUs. So you learn how each block is made up of an attention layer and a feed-forward network. The model then uses the output vectors of the transformer blocks and passes them to the last component, the language modeling head, which generates the output token.

    <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-05.png" width="800">



The magic of LLMs actually comes from two parts: One, the transformer architecture, which is well worth learning. And second, all the incredibly rich data that the models learn from. So while the magic of LLMs comes not just from the transformer architecture, but also from the data, having a solid intuition of what this architecture is doing will give you better intuitions about why they behave in certain ways, as well as how to use them.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Understanding Language Models: Language as a Bag-of-Words

In the next several sections, you will learn the evolution of how language has been represented numerically. We'll start with **Bag of Words**, an algorithm that represents words as large sparse vectors or arrays of numbers, which simply record the presence of words. Then **Word2Vec**, whose word representation capture the meaning of words in the context of a few neighboring words. Finally, **Transformers**, whose dense vectors capture the meaning of words in the context of a sentence or a paragraph.

Although we will explore a more recent history of language AI and language models, it is important to understand where we started. Earlier techniques such as Bag-of-Words and Word2Vec have been arguably the foundation of what we're using today. Although they lack contextualized representations, they are often a good baseline to start with. We call these **non-transformer models** since today's models are typically powered solely by transformer models, in contrast to these strong baselines.

Then we have **encoder-only models**, which are great at representing language in numerical representations. In contrast, **decoder-only models** are generative in their nature. Their main usage is to generate text. And finally, we have **encoder-decoder models** that attempt to get the best of both worlds. You will learn about these non-transformer models, encoders, decoders and how they relate, as we explore this more recent history of language AI.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-06.png" width="800">

Language however, is a tricky concept for computers. Text is unstructured in nature and loses its meaning when represented by zeros and ones or individual characters. As a result, throughout the history of language AI, there has been a large focus on representing language in a structured manner so that it can more easily be used by computers. From generating text, to creating numerical representations, and classifying textual inputs — these are just a few of the numerous tasks you can do with language AI.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-07.png" width="800">

At the start of this Language AI field, the focus was mainly on representing language to analyze unstructured data. A first and still very relevant method is by representing language as a bag-of-words. Imagine you have some input text, `"That is a cute dog"`. To represent this sentence you can break it up into smaller pieces. To do so, you split the text into words by separating them through a whitespace. This process of converting the input text into pieces is called *tokenization*, and each individual word is called a token. Note that a token can be any even smaller than an entire word, but we will go through tokenization in the future sections in more detail. You can perform the same tokenization process with another document, `"My cat is cute"`. Although similar documents, they do contain different words and have a different meaning. Now that you are left with two sets of tokens, you can create something called a *vocabulary*. This vocabulary contains all unique words or tokens found in both input documents. As such, the vocabulary will contain fewer words than the amount of tokens that were generated. And we typically refer to the number of tokens or words in the vocabulary as the vocabulary size.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-08.png" width="800">

To then represent one of our inputs with numerical values, let's focus a bit on our second input, `"My cat is cute"`. This input has four tokens that match with some of the words in the vocabulary, but not all. You can then count how often a certain token appears in the vocabulary that we already created. In this case, each token appears once. However, you will also have to take note of the words in the vocabulary that do not appear in the input. A sentence not only gives meaning to the words it contains, but also the words it doesn't. Now that you have counted how often the words or tokens of our input appear in the vocabulary, you have created your numerical representation. This is called a bag-of-words, and does nothing more than counting individual words that appear in the vocabulary. Therefore, the numerical representation of `"My cat is cute"` is `0101011`, in that specific order. The order is important as it allows us to compare different sentences to one another. In practice, we call this a vector representation, a list of numerical values that represents the input. In this example, these values are counts and have an explicit meaning, namely, the number of times a word in the vocabulary appears in the input. Vector representations, in more complex and advanced models, typically do not have such an intuitive meaning, and have values between 0 and 1. In the next section, we'll learn how these vector embeddings are created.

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/Generative-AI/How-Transformer-LLMs-Work/data/images/How_Transformer_LLMs_Work-09.png" width="800">

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **[DeepLearning.AI: How Transformer LLMs Work Course](https://www.deeplearning.ai/courses/how-transformer-llms-work)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

<img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">

<table>
  <tr>
    <td align="center">
      <img src="https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/refs/heads/main/NLP/NLP-Foundations/data/images/Word_Embeddings-01.png" width="400">
    </td>
  </tr>
  <tr>
    <td align="center">
      <em>Figure 1: Word Embeddings</em>
    </td>
  </tr>
</table>

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)